# likable-river — Statistical EDA (full train DEP, n=2,085,047)
Distribution shape, factor significance + effect sizes, correlations,train-vs-ranking shift tests, tail dominance. Ends with assertions.

In [1]:
import glob
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA = '../../data'
FIG = 'figures'
train = ds.dataset(sorted(glob.glob(f'{DATA}/training_*.parquet')), format='parquet')
rank = ds.dataset([f'{DATA}/ranking.parquet'], format='parquet')
DEP = (ds.field('PHASE_mvt') == 'DEP')
pd.set_option('display.width', 180)


## 1. Target shape — skew, kurtosis, log-normality

In [2]:
y = train.to_table(columns=['TAXITIME_SEC_mvt'], filter=DEP).to_pandas()['TAXITIME_SEC_mvt'].to_numpy(dtype=float)
yp = y[y > 0]
print(f"n={len(y)}, skew={stats.skew(y):.2f}, kurtosis={stats.kurtosis(y):.1f}")
print(f"log1p: skew={stats.skew(np.log1p(yp)):.2f}, kurtosis={stats.kurtosis(np.log1p(yp)):.1f}")
rng = np.random.default_rng(0)
s = rng.choice(yp, 20000, replace=False)
k2, p = stats.normaltest(np.log1p(s))
print(f"log-normality normaltest on log1p(sample 20k): stat={k2:.0f}, p={p:.2e} (reject => not lognormal)")
qs = [50, 75, 90, 95, 99, 99.9, 99.99]
print('quantiles:', {q: round(float(np.quantile(y, q/100)), 1) for q in qs})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y[y < 4000], bins=100); axes[0].set_title('taxi-out (<4000s)')
axes[1].hist(np.log1p(yp), bins=100); axes[1].set_title('log1p(taxi-out)')
fig.savefig(f'{FIG}/stat_shape.png', dpi=80); plt.close(fig); print('saved stat_shape')


n=2085047, skew=48.44, kurtosis=7016.3
log1p: skew=-0.61, kurtosis=6.6
log-normality normaltest on log1p(sample 20k): stat=2531, p=0.00e+00 (reject => not lognormal)
quantiles: {50: 912.0, 75: 1188.0, 90: 1471.0, 95: 1684.0, 99: 2339.0, 99.9: 4501.0, 99.99: 10711.8}


saved stat_shape


## 2. Factor significance — Kruskal-Wallis + eta-squared (effect size, not just p)

In [3]:
cols = ['TAXITIME_SEC_mvt', 'ADEP_mvt', 'AIRCRAFT_TYPE_mvt', 'WK_TBL_CAT_flt',
        'MARKET_SEGMENT_flt', 'FLIGHT_TYPE_flt', 'RUNWAY_mvt', 'BLOCK_TIME_UTC_mvt',
        'AIRCRAFT_OPERATOR_flt']
d = train.to_table(columns=cols, filter=DEP).to_pandas()
d['month'] = d['BLOCK_TIME_UTC_mvt'].dt.month
d['hour'] = d['BLOCK_TIME_UTC_mvt'].dt.hour
d['dow'] = d['BLOCK_TIME_UTC_mvt'].dt.dayofweek
yy = d['TAXITIME_SEC_mvt'].to_numpy(dtype=float)
grand = yy.mean()
print(f"{'factor':<18}{'levels':>7}{'KW p':>10}{'eta2':>8}  (eta2 = between-SS/total-SS)")
for c in ['ADEP_mvt', 'hour', 'dow', 'month', 'WK_TBL_CAT_flt', 'MARKET_SEGMENT_flt',
          'FLIGHT_TYPE_flt', 'AIRCRAFT_TYPE_mvt', 'AIRCRAFT_OPERATOR_flt', 'RUNWAY_mvt']:
    g = d.groupby(c, observed=True)['TAXITIME_SEC_mvt']
    kw = stats.kruskal(*[v.to_numpy(dtype=float) for _, v in g])
    ss_b = ((g.mean() - grand) ** 2 * g.size()).sum()
    ss_t = ((yy - grand) ** 2).sum()
    print(f"{c:<18}{g.ngroups:>7}{kw.pvalue:>10.2e}{ss_b/ss_t:>8.4f}")


factor             levels      KW p    eta2  (eta2 = between-SS/total-SS)


ADEP_mvt               10  0.00e+00  0.1121


hour                   24  0.00e+00  0.0061


dow                     7 2.68e-192  0.0003


month                  12  0.00e+00  0.0007


WK_TBL_CAT_flt          4  0.00e+00  0.0357


MARKET_SEGMENT_flt      8  0.00e+00  0.0239


FLIGHT_TYPE_flt         5  0.00e+00  0.0022


AIRCRAFT_TYPE_mvt     269  0.00e+00  0.0676


AIRCRAFT_OPERATOR_flt    676  0.00e+00  0.1152
RUNWAY_mvt             53  0.00e+00  0.1331


## 3. Numeric correlations (Spearman, robust to tail) + delay anatomy

In [4]:
c2 = ['TAXITIME_SEC_mvt', 'MVT_TIME_UTC_mvt', 'BLOCK_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt',
      'AOBT_3_flt', 'LOBT_flt', 'IOBT_flt', 'EOBT_1_flt']
t = train.to_table(columns=c2, filter=DEP).to_pandas()
for a, b, n in [('MVT_TIME_UTC_mvt', 'AOBT_3_flt', 'm_aobt'), ('MVT_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt', 'm_sched'),
                ('AOBT_3_flt', 'SCHED_TIME_UTC_mvt', 'aobt_sched'), ('LOBT_flt', 'SCHED_TIME_UTC_mvt', 'lobt_sched'),
                ('EOBT_1_flt', 'SCHED_TIME_UTC_mvt', 'eobt_sched'), ('BLOCK_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt', 'blk_sched')]:
    t[n] = (t[a] - t[b]).dt.total_seconds()
keep = t[['TAXITIME_SEC_mvt', 'm_aobt', 'm_sched', 'aobt_sched', 'lobt_sched', 'eobt_sched', 'blk_sched']].dropna()
print('spearman vs target:')
print(keep.corr(method='spearman')['TAXITIME_SEC_mvt'].round(4).to_string())
print('pearson vs target:')
print(keep.corr(method='pearson')['TAXITIME_SEC_mvt'].round(4).to_string())
print('spearman-p of m_aobt:', f"{stats.spearmanr(keep['m_aobt'], keep['TAXITIME_SEC_mvt']).pvalue:.2e}")


spearman vs target:


TAXITIME_SEC_mvt    1.0000
m_aobt              0.6150
m_sched             0.3883
aobt_sched          0.1530
lobt_sched          0.0139
eobt_sched          0.0002
blk_sched          -0.0170
pearson vs target:
TAXITIME_SEC_mvt    1.0000
m_aobt              0.5358
m_sched             0.2049
aobt_sched          0.1138
lobt_sched          0.0331
eobt_sched          0.0375
blk_sched           0.0163


spearman-p of m_aobt: 0.00e+00


## 4. Shift tests — is ranking different? (KS on samples + median CIs)

In [5]:
tr = train.to_table(columns=['MVT_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt', 'AOBT_3_flt'],
    filter=DEP).to_pandas().dropna()
rk = rank.to_table(columns=['MVT_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt', 'AOBT_3_flt'],
    filter=DEP).to_pandas().dropna()
rng = np.random.default_rng(1)
for n, (a, b) in {'m_aobt': ('MVT_TIME_UTC_mvt', 'AOBT_3_flt'),
                  'm_sched': ('MVT_TIME_UTC_mvt', 'SCHED_TIME_UTC_mvt')}.items():
    x = ((tr[a] - tr[b]).dt.total_seconds()).to_numpy()
    z = ((rk[a] - rk[b]).dt.total_seconds()).to_numpy()
    xs = rng.choice(x, 100000, replace=False); zs = rng.choice(z, 100000, replace=False)
    ks = stats.ks_2samp(xs, zs)
    print(f"{n}: train med={np.median(x):.0f} rank med={np.median(z):.0f} "
          f"KS D={ks.statistic:.4f} p={ks.pvalue:.2e} "
          f"(D<0.02 => same shape, median drift only)")


m_aobt: train med=958 rank med=978 KS D=0.0294 p=4.16e-38 (D<0.02 => same shape, median drift only)


m_sched: train med=1389 rank med=1497 KS D=0.0416 p=1.77e-75 (D<0.02 => same shape, median drift only)


## 5. Tail dominance — who owns the squared error?

In [6]:
amed_full = d.groupby('ADEP_mvt', observed=True)['TAXITIME_SEC_mvt'].median()
for name, pred in [('global-median', np.full_like(yy, np.median(yy))),
                   ('airport-median', d['ADEP_mvt'].map(amed_full).to_numpy())]:
    e = (yy - pred) ** 2
    tot = e.sum()
    print(f"{name} in-sample RMSE={np.sqrt(e.mean()):.1f}")
    sh = d.assign(e=e).groupby('ADEP_mvt', observed=True)['e'].sum() / tot * 100
    print(sh.round(1).sort_values(ascending=False).to_string())
LIRF_SHARE = float((d.assign(e=(yy - d['ADEP_mvt'].map(amed_full).to_numpy()) ** 2)
    .groupby('ADEP_mvt', observed=True)['e'].sum() / ((yy - d['ADEP_mvt'].map(amed_full).to_numpy()) ** 2).sum() * 100)['LIRF'])
print(f"LIRF airport-median SSE share (full train): {LIRF_SHARE:.1f}% "
      f"(vs 59.6% on Jan+Jul holdout — tail concentration varies by month)")
print('row share by airport (%):')
print((d['ADEP_mvt'].value_counts(normalize=True) * 100).round(1).to_string())
e0 = (yy - np.median(yy)) ** 2
for q in [90, 95, 99, 99.9]:
    th = np.quantile(yy, q / 100)
    m = yy >= th
    print(f"top {100-q:.1f}% (>= {th:.0f}s): rows={m.mean()*100:.2f}% SSE share={e0[m].sum()/e0.sum()*100:.1f}%")


global-median in-sample RMSE=552.1
ADEP_mvt
LIRF    46.9
EGLL    14.4
LTFM     8.7
LFPG     8.2
EHAM     4.6
EDDF     4.0
LSZH     3.8
LEMD     3.6
LEBL     3.2
EDDM     2.7
airport-median in-sample RMSE=519.5


ADEP_mvt
LIRF    51.5
LTFM     9.3
LFPG     8.9
EGLL     7.7
EDDF     4.5
EHAM     4.5
LEMD     3.7
LEBL     3.6
LSZH     3.6
EDDM     2.8


LIRF airport-median SSE share (full train): 51.5% (vs 59.6% on Jan+Jul holdout — tail concentration varies by month)
row share by airport (%):


ADEP_mvt
LTFM    13.1
EHAM    11.9
LFPG    11.5
EGLL    11.5
EDDF    11.0
LEMD    10.2
LEBL     8.6
EDDM     8.0
LIRF     7.7
LSZH     6.5
top 10.0% (>= 1471s): rows=10.00% SSE share=78.5%
top 5.0% (>= 1684s): rows=5.02% SSE share=71.5%
top 1.0% (>= 2339s): rows=1.00% SSE share=58.2%
top 0.1% (>= 4501s): rows=0.10% SSE share=46.0%


## 6. Assertions

In [7]:
assert abs(stats.skew(y)) > 2, 'expected heavy right skew'
assert 45 <= LIRF_SHARE <= 65, LIRF_SHARE
print('ALL STATISTICAL ASSERTIONS PASS')


ALL STATISTICAL ASSERTIONS PASS
